# DSCI 552 — Homework 6
**Danmu Agudanmu**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (confusion_matrix, roc_curve, roc_auc_score,
                             accuracy_score, ConfusionMatrixDisplay)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

SEED = 42
DATA = "../Homework 6 Data/"
pd.set_option("display.max_columns", 30)

## 1. Tree-Based Methods

### (a) Load data
The first 20 lines of each csv are the license header. Missing values are coded as `na`.

In [ ]:
train = pd.read_csv(DATA + "aps_failure_training_set.csv", skiprows=20, na_values="na")
test = pd.read_csv(DATA + "aps_failure_test_set.csv", skiprows=20, na_values="na")

y_train = (train.pop("class") == "pos").astype(int)
y_test = (test.pop("class") == "pos").astype(int)

print("train:", train.shape, " test:", test.shape)

### (b) Data preparation

#### i. Missing values

Common approaches include deleting incomplete rows, mean/median imputation, KNN or iterative imputation, and adding missing-value indicators. Deleting rows would discard too much data here. I use the median because many features are skewed.

In [ ]:
miss = train.isna().mean().sort_values(ascending=False)
print("rows with at least one missing value: %.1f%%" % (100 * train.isna().any(axis=1).mean()))
print("features with >50%% missing: %d" % (miss > 0.5).sum())
miss.head(10).round(3)

I use column medians to fill missing values, separately for training and test as noted in the assignment. For cross-validation, imputation is fitted within each training fold.

In [ ]:
X_train = train.fillna(train.median())
X_test = test.fillna(test.median())
print(X_train.isna().sum().sum(), X_test.isna().sum().sum())

#### ii. Coefficient of variation

In [ ]:
cv = (X_train.std() / X_train.mean()).replace([np.inf, -np.inf], np.nan)
cv = cv.sort_values(ascending=False)
print("features with zero variance:", list(X_train.columns[X_train.std() == 0]))
cv_table = cv.to_frame("CV")
cv_table.T

#### iii. Correlation matrix

In [ ]:
corr = X_train.corr()
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-1, vmax=1,
            xticklabels=10, yticklabels=10, ax=ax)
ax.set_title("Correlation matrix of the 170 features (training set)")
plt.show()

#### iv. Scatter plots and box plots for the ⌊√170⌋ = 13 features with highest CV

The counters span several orders of magnitude, so they are plotted on a log(1 + x) scale. Positives are drawn on top of negatives.

In [ ]:
k = int(np.floor(np.sqrt(170)))
top = cv.index[:k].tolist()
print(k, top)

plot_df = np.log1p(X_train[top]).assign(cls=y_train.map({0: "neg", 1: "pos"}))
g = sns.pairplot(plot_df, vars=top, hue="cls", hue_order=["neg", "pos"],
                 palette={"neg": "tab:blue", "pos": "tab:orange"},
                 plot_kws=dict(s=4, alpha=0.4, linewidth=0, rasterized=True),
                 diag_kind="kde", height=1.4, corner=True)
g.fig.suptitle("Pairwise scatter plots, log(1 + x)", y=1.01)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 7, figsize=(20, 7))
for ax, f in zip(axes.ravel(), top):
    sns.boxplot(data=plot_df, x="cls", y=f, hue="cls", order=["neg", "pos"], ax=ax,
                palette={"neg": "tab:blue", "pos": "tab:orange"}, legend=False, fliersize=1)
    ax.set_xlabel(""); ax.set_title(f)
axes.ravel()[-1].axis("off")
fig.suptitle("Box plots by class, log(1 + x)")
plt.tight_layout(); plt.show()

The plots show substantial overlap between classes. A few features have different distributions, but high CV alone does not show that a feature is useful for prediction, and these plots cannot establish statistical significance.

#### v. Class balance

In [ ]:
print(pd.DataFrame({"train": y_train.value_counts(), "test": y_test.value_counts()})
      .rename(index={0: "neg", 1: "pos"}))

Training has 59,000 negative and 1,000 positive cases; test has 15,625 negative and 375 positive cases. The data is highly imbalanced.

### (c) Random forest without class weighting

The following function reports confusion matrices, ROC curves, AUC, and classification error.

In [ ]:
results = {}

def evaluate(model, name, sets):
    fig, axes = plt.subplots(1, len(sets) + 1, figsize=(5 * (len(sets) + 1), 4))
    rows = []
    for ax, (split, X, y) in zip(axes[1:], sets):
        p = model.predict_proba(X)[:, 1]
        pred = (p >= 0.5).astype(int)
        cm = confusion_matrix(y, pred)
        auc = roc_auc_score(y, p)
        err = 1 - accuracy_score(y, pred)
        fpr, tpr, _ = roc_curve(y, p)
        axes[0].plot(fpr, tpr, label=f"{split} (AUC = {auc:.4f})")
        ConfusionMatrixDisplay(cm, display_labels=["neg", "pos"]).plot(ax=ax, colorbar=False, values_format="d")
        ax.set_title(f"{split} confusion matrix")
        rows.append({"model": name, "set": split, "AUC": auc, "misclassification": err,
                     "TN": cm[0, 0], "FP": cm[0, 1], "FN": cm[1, 0], "TP": cm[1, 1]})
    axes[0].plot([0, 1], [0, 1], "k--", lw=0.8)
    axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title=f"ROC — {name}")
    axes[0].legend(loc="lower right")
    plt.tight_layout(); plt.show()
    out = pd.DataFrame(rows).set_index(["model", "set"])
    results[name] = out
    return out

In [ ]:
rf = RandomForestClassifier(n_estimators=300, oob_score=True, n_jobs=-1, random_state=SEED)
rf.fit(X_train, y_train)

sets = [("train", X_train, y_train), ("test", X_test, y_test)]
evaluate(rf, "RF", sets).round(4)

In [ ]:
rf_oob_err = 1 - rf.oob_score_
rf_test_err = results["RF"].loc[("RF", "test"), "misclassification"]
print(f"OOB error: {rf_oob_err:.4f}   test error: {rf_test_err:.4f}")

Compare OOB and test error above. Training error is optimistic; false negatives matter more than accuracy alone for this imbalanced dataset.

### (d) Random forest with class weighting

Imbalance can be handled using class weights, balanced bootstrap samples, or resampling. Here I use `balanced_subsample` and compare it with (c).

In [ ]:
rf_bal = RandomForestClassifier(n_estimators=300, oob_score=True, n_jobs=-1,
                                class_weight="balanced_subsample", random_state=SEED)
rf_bal.fit(X_train, y_train)
evaluate(rf_bal, "RF balanced", sets).round(4)

In [ ]:
rf_bal_oob_err = 1 - rf_bal.oob_score_
rf_bal_test_err = results["RF balanced"].loc[("RF balanced", "test"), "misclassification"]
print(f"OOB error: {rf_bal_oob_err:.4f}   test error: {rf_bal_test_err:.4f}")
pd.concat([results["RF"], results["RF balanced"]]).round(4)

Compare the test AUC, error, and false negatives with (c). Class weighting does not necessarily improve recall at the default 0.5 threshold.

### (e) XGBoost with L1 regularization

I tune `reg_alpha` using five-fold stratified cross-validation. XGBoost uses logistic loss and L1-regularized leaf weights; its usual tree splits are univariate, not logistic-regression splits at each node.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
alphas = [0, 0.01, 0.1, 1, 10, 100]

xgb_base = dict(objective="binary:logistic", n_estimators=200, max_depth=4,
                learning_rate=0.1, tree_method="hist", n_jobs=-1,
                eval_metric="logloss", random_state=SEED)

xgb_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("xgb", XGBClassifier(**xgb_base))
])
gs_xgb = GridSearchCV(xgb_pipe, {"xgb__reg_alpha": alphas}, cv=skf,
                      scoring={"auc": "roc_auc", "acc": "accuracy"}, refit="auc", n_jobs=1)
gs_xgb.fit(train, y_train)

cvres = pd.DataFrame(gs_xgb.cv_results_)
cv_table_xgb = pd.DataFrame({"alpha": alphas,
                             "CV AUC": cvres["mean_test_auc"].values,
                             "CV error": 1 - cvres["mean_test_acc"].values})
print("best alpha:", gs_xgb.best_params_["xgb__reg_alpha"])
cv_table_xgb.round(5)

In [ ]:
xgb = gs_xgb.best_estimator_
xgb_cv_err = 1 - cvres.loc[gs_xgb.best_index_, "mean_test_acc"]
out = evaluate(xgb, "XGB", sets)
print(f"5-fold CV error: {xgb_cv_err:.4f}   test error: {out.loc[('XGB','test'),'misclassification']:.4f}")
out.round(4)

The table above gives cross-validation AUC and error for each value of α. Compare the selected model's CV error with its test error; the test AUC is reported above.

### (f) SMOTE + XGBoost

Imputation and SMOTE are fitted only on the training portion of each CV fold. This avoids using validation data to construct synthetic training samples.

In [ ]:
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("smote", SMOTE(random_state=SEED)),
    ("xgb", XGBClassifier(**xgb_base))
])

gs_smote = GridSearchCV(pipe, {"xgb__reg_alpha": alphas}, cv=skf,
                        scoring={"auc": "roc_auc", "acc": "accuracy"}, refit="auc", n_jobs=1)
gs_smote.fit(train, y_train)

cvres_s = pd.DataFrame(gs_smote.cv_results_)
cv_table_smote = pd.DataFrame({"alpha": alphas,
                               "CV AUC": cvres_s["mean_test_auc"].values,
                               "CV error": 1 - cvres_s["mean_test_acc"].values})
print("best alpha:", gs_smote.best_params_["xgb__reg_alpha"])
cv_table_smote.round(5)

In [ ]:
xgb_smote = gs_smote.best_estimator_
smote_cv_err = 1 - cvres_s.loc[gs_smote.best_index_, "mean_test_acc"]
out = evaluate(xgb_smote, "XGB + SMOTE", sets)
print(f"5-fold CV error: {smote_cv_err:.4f}   test error: {out.loc[('XGB + SMOTE','test'),'misclassification']:.4f}")
out.round(4)

#### Summary of all models

In [ ]:
summary = pd.concat(results.values())
summary.round(4)

**Comparison.** SMOTE tends to detect more positive cases but can also increase false alarms. Compare the test-set FN, FP, AUC, and error above. CV is performed before SMOTE on each held-out fold, not on an already-oversampled dataset.

## 2. ISLR 6.6.3

As the Lasso constraint $\sum_j |\beta_j| \le s$ becomes less restrictive:

**(a) (iv)** Training RSS decreases: the feasible set expands.

**(b) (ii)** Test RSS generally decreases then increases due to the bias–variance tradeoff.

**(c) (iii)** Variance increases as the model becomes more flexible.

**(d) (iv)** Squared bias decreases with flexibility.

**(e) (v)** Irreducible error stays constant.

## 3. ISLR 6.6.5

Setup: $n=p=2$, $x_{11}=x_{12}$, $x_{21}=x_{22}$, $y_1+y_2=0$, $x_{11}+x_{21}=0$, $x_{12}+x_{22}=0$, so $\hat\beta_0=0$.
Write $x_1 = x_{11}=x_{12}$ and $x_2 = x_{21}=x_{22}$, so $x_2=-x_1$ and $y_2=-y_1$.

**(a)** Ridge:
$$\min_{\beta_1,\beta_2}\; (y_1-\beta_1x_{11}-\beta_2x_{12})^2 + (y_2-\beta_1x_{21}-\beta_2x_{22})^2 + \lambda(\beta_1^2+\beta_2^2)$$

**(b)** With the setup, the RSS only depends on $\beta_1+\beta_2$:
$$f(\beta_1,\beta_2) = (y_1-(\beta_1+\beta_2)x_1)^2 + (y_2-(\beta_1+\beta_2)x_2)^2 + \lambda(\beta_1^2+\beta_2^2)$$
Setting the partial derivatives to zero:
$$\frac{\partial f}{\partial\beta_1} = -2x_1(y_1-(\beta_1+\beta_2)x_1) - 2x_2(y_2-(\beta_1+\beta_2)x_2) + 2\lambda\beta_1 = 0$$
$$\frac{\partial f}{\partial\beta_2} = -2x_1(y_1-(\beta_1+\beta_2)x_1) - 2x_2(y_2-(\beta_1+\beta_2)x_2) + 2\lambda\beta_2 = 0$$
The first two terms are identical in both equations. Subtracting gives $2\lambda(\beta_1-\beta_2)=0$, so for $\lambda>0$, $\hat\beta_1=\hat\beta_2$.

**(c)** Lasso:
$$\min_{\beta_1,\beta_2}\; (y_1-\beta_1x_{11}-\beta_2x_{12})^2 + (y_2-\beta_1x_{21}-\beta_2x_{22})^2 + \lambda(|\beta_1|+|\beta_2|)$$

**(d)** Using $x_2=-x_1$, $y_2=-y_1$, the RSS is $2\,(y_1-(\beta_1+\beta_2)x_1)^2$, a function of $\beta_1+\beta_2$ only. Its contours are lines $\beta_1+\beta_2=c$, parallel to the edges of the lasso diamond $|\beta_1|+|\beta_2|\le s$.
In the constrained form, the least-squares solution needs $\beta_1+\beta_2 = y_1/x_1$; if that is outside the diamond, the RSS contour first touches the diamond along a whole edge, not a single corner. So the solution is not unique:
$$\hat\beta_1+\hat\beta_2 = s,\; \hat\beta_1,\hat\beta_2\ge 0 \quad\text{(if } y_1/x_1>0\text{)}$$
$$\hat\beta_1+\hat\beta_2 = -s,\; \hat\beta_1,\hat\beta_2\le 0 \quad\text{(if } y_1/x_1<0\text{)}$$
i.e. every point on the segment of the diamond's edge in that quadrant is a lasso solution. (In the penalized form the same holds: any $\beta_1,\beta_2$ with the same sign and the optimal sum give the same RSS and the same $|\beta_1|+|\beta_2|$.)

## 4. ISLR 8.4.5

Estimates of $P(\text{Red}\mid X)$ from 10 bootstrap trees: 0.1, 0.15, 0.2, 0.2, 0.55, 0.6, 0.6, 0.65, 0.7, 0.75.

In [ ]:
p = np.array([0.1, 0.15, 0.2, 0.2, 0.55, 0.6, 0.6, 0.65, 0.7, 0.75])
print("votes for Red:", (p > 0.5).sum(), "of", len(p), "->", "Red" if (p > 0.5).sum() > len(p) / 2 else "Green")
print("average probability:", p.mean(), "->", "Red" if p.mean() > 0.5 else "Green")

- **Majority vote:** 6 of 10 trees predict Red, so the result is **Red**.
- **Average probability:** $\bar p=0.45$, so the result is **Green**.

## 5. ISLR 9.7.3

**(a)** Observations:

In [ ]:
X93 = np.array([[3, 4], [2, 2], [4, 4], [1, 4], [2, 1], [4, 3], [4, 1]])
c93 = np.array(["red"] * 4 + ["blue"] * 3)

def base_plot(ax, title):
    ax.scatter(X93[:, 0], X93[:, 1], c=c93, s=60, zorder=3)
    for i, (a, b) in enumerate(X93, 1):
        ax.annotate(str(i), (a, b), textcoords="offset points", xytext=(5, 5))
    ax.set(xlim=(0, 5), ylim=(0, 5), xlabel="$X_1$", ylabel="$X_2$", title=title)
    ax.set_aspect("equal"); ax.grid(alpha=0.3)

xs = np.linspace(0, 5, 100)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
base_plot(axes[0], "(a) observations")

base_plot(axes[1], "(b), (d) maximal margin hyperplane")
axes[1].plot(xs, xs - 0.5, "k-", label="$0.5 - X_1 + X_2 = 0$")
axes[1].plot(xs, xs, "k--", lw=0.8, label="margins")
axes[1].plot(xs, xs - 1, "k--", lw=0.8)
sv = X93[[1, 2, 4, 5]]
axes[1].scatter(sv[:, 0], sv[:, 1], s=200, facecolors="none", edgecolors="k", label="support vectors")
axes[1].legend(loc="upper left", fontsize=8)

base_plot(axes[2], "(g) a non-optimal separating hyperplane")
axes[2].plot(xs, xs - 0.1, "g-", label="$0.1 - X_1 + X_2 = 0$")
axes[2].legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

**(b)** The closest points of the two classes are (2,2), (4,4) (red) and (2,1), (4,3) (blue). Reds lie on $X_2 = X_1$, blues on $X_2 = X_1 - 1$. The maximal margin hyperplane lies halfway between:
$$X_2 = X_1 - 0.5 \quad\Longleftrightarrow\quad 0.5 - X_1 + X_2 = 0$$

**(c)** Classify to Red if $0.5 - X_1 + X_2 > 0$, and to Blue otherwise. So $\beta_0 = 0.5$, $\beta_1 = -1$, $\beta_2 = 1$.

**(d)** Shown in the middle plot (dashed lines). The margin is the distance from a support vector to the hyperplane: $|0.5 - 2 + 2|/\sqrt{2} = 0.5/\sqrt 2 \approx 0.354$.

**(e)** Support vectors: observations 2 (2,2), 3 (4,4), 5 (2,1), 6 (4,3).

**(f)** Observation 7 (4,1) is not a support vector and is well outside the margin, so a slight move does not change the hyperplane.

**(g)** For example $0.1 - X_1 + X_2 = 0$ (right plot). It separates the classes but sits much closer to the red points (2,2) and (4,4), so its margin is smaller ($0.1/\sqrt2 \approx 0.07$).

**(h)** Add a red point at (3,1):

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
base_plot(ax, "(h) not separable after adding (3, 1)")
ax.scatter([3], [1], c="red", s=60, marker="s", zorder=3)
ax.annotate("8", (3, 1), textcoords="offset points", xytext=(5, 5))
plt.show()

The red point (3,1) lies on the line segment joining two blue points, (2,1) and (4,1). No linear boundary can separate it from both of them.